In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
import numpy as np

In [ ]:
# change working directory to project root
import os

os.chdir('/Users/fazedreaper/Documents/Software-Projects/Python/Campus Gym Capacity Predictor')

# read the cleaned csv file
gym_df = pd.read_csv('data/processed/rutgers_gym_cleaned.csv')

# analyze the data
gym_df.head()
print(gym_df.columns.tolist())

In [ ]:
# initialize the label encoder
gym_encoder = LabelEncoder()
day_encoder = LabelEncoder()
weather_encoder = LabelEncoder()
capacity_encoder = LabelEncoder()


# convert gym names into numbers 
gym_df['gym_name_encoded'] = gym_encoder.fit_transform(gym_df['gym_name'])

# convert day names into numbers 
gym_df['day_of_week_encoded'] = day_encoder.fit_transform(gym_df['day_of_week'])

# convert weathers into numbers 
gym_df['weather_encoded'] = weather_encoder.fit_transform(gym_df['weather'])

# convert capacity into numbers -> target 
gym_df['capacity_label_encoded'] = capacity_encoder.fit_transform(gym_df['capacity_label'])

# check if encoding worked
print(gym_df[['gym_name', 'gym_name_encoded', 'day_of_week', 'day_of_week_encoded', 'capacity_label', 'capacity_label_encoded']].head())


In [ ]:
# select columns as features for the model
features = [
    'gym_name_encoded',
    'day_of_week_encoded',
    'hour_24',
    'is_weekend',
    'semester_week',
    'is_exam_week',
    'weather_encoded'
]

# define X for the input
X = gym_df[features]

# define y for the target and display the shapes for each
y = gym_df['capacity_label_encoded']

print('Input shape:', X.shape)
print('Target shape:', y.shape)

In [ ]:
# split data training:testing -> 80%:20%
# random_state=42 ensures consistent result every run
# display the rows for training and testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

In [ ]:
# initialize Random Forest
# n_estimators=100 -> 100 decision trees and votes on result
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# train model on test data
rf_model.fit(X_train, y_train)

print('Model trained successfully')


In [ ]:
# use the trained model to predict test data
y_pred = rf_model.predict(X_test)

# calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

# calculate MAE (average diff between pred. and actual)
mae = mean_absolute_error(y_test, y_pred)

# calculate RMSE 
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print('Accuracy:', round(accuracy * 100, 2), '%')
print('MAE:', round(mae, 3))
print('RMSE:', round(rmse, 3))

In [ ]:
# feature importance scores from model
importances = rf_model.feature_importances_

# plot bar chart
plt.figure(figsize=(10, 6))
plt.bar(features, importances)

# title and labels
plt.title('Feature Importance- Random Forest')
plt.xlabel('Feature')
plt.ylabel('Importance Score')

# emove overlapping with x-axis 
plt.xticks(rotation=45, ha='right')

# space adjustment
plt.tight_layout()

# display the plot
plt.show()

In [ ]:
#prediction function to predict the capacity of a gym
def predict_capacity(gym_name, day_of_week, hour_24, is_weekend, semester_week, is_exam_week, weather):
    #use encoders to convert from text to numbers
    gym_encoded = gym_encoder.transform([gym_name][0])
    day_encoded = day_encoder.transform([day_of_week][0])
    weather_enecoded = weather_encoder.transform([weather][0])

    #input array in same order as features
    input_array = [[gym_encoded, day_encoded, hour_24, is_weekend, semester_week, is_exam_week, weather_encoded]]

    #make prediction
    prediction_encoded = rf_model.predict(input_data)[0]

    #convert prediction number into label
    prediction_label = capacity_encoder.inverse_transform(prediction_encoded.reshape(1,))[0]

    return prediction_label

In [ ]:
#Test 1: College Ave, Monday evening, week 7 of semester, weekday, normal week, clear weather
result1 = predict_capacity(
    gym_name='College Avenue Gym',
    day_of_week='Monday',
    hour_24=18,
    is_weekend=False,
    semester_week=7,
    is_exam_week=False,
    weather='clear'
)
print("Monday 6pm at College Ave:", result1)